# Grammars and Parsing

## MSA 8700 — Module 8: NLP and Text Processing

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what a **Context-Free Grammar (CFG)** is and how it defines sentence structure
2. Write CFG rules using **NLTK's grammar notation**
3. Parse sentences with NLTK's **ChartParser** and interpret the resulting **parse trees**
4. Build grammars that handle **real-world complexity**: optional elements, alternatives, and recursion
5. Understand **Definite Clause Grammars (DCGs)** in Prolog as an alternative formalism
6. See how rule-based parsing fits into **agentic AI workflows** alongside LLMs

---

# Part 1: Why Rule-Based Grammars Still Matter

## LLMs Learn Syntax Implicitly — So Why Bother?

Large Language Models learn grammar, syntax, and structure from massive amounts of text. They can parse, generate, and manipulate language with remarkable fluency. So why study explicit, rule-based grammars?

There are three important scenarios where explicit grammars outperform or complement LLMs:

| Scenario | Why Explicit Grammars Help |
|----------|---------------------------|
| **Hard constraints** | Only accept input that matches a specific, pre-defined pattern. No hallucination, no approximation — if it doesn't parse, it's rejected. |
| **Explainable parsing** | The parse tree shows *exactly* how the input was interpreted. Every decision is traceable — critical for auditing and debugging. |
| **Teaching and testing** | Grammars make linguistic structure visible. They're the foundation of computational linguistics and help build intuition about how language works. |

### The Agentic AI Angle

In an agent system, a **command interpreter** component can use a small grammar to parse a constrained set of user commands:

```
User input:  "cancel the subscription"
                    ↓
CFG Parser:  Verb = "cancel",  Object = "subscription"
                    ↓
Tool Call:   cancel_subscription()
```

The grammar **guarantees** that only valid commands are accepted. The LLM can sit on top to handle paraphrases and fallback cases ("I don't want this anymore" → "cancel the subscription"), but the actual command execution goes through a deterministic parser.

---

# Part 2: Context-Free Grammars — The Fundamentals

## What Is a Context-Free Grammar?

A **Context-Free Grammar (CFG)** is a set of rules that define how symbols can be combined to form valid strings (sentences). It has four components:

| Component | Symbol | Description | Example |
|-----------|--------|-------------|--------|
| **Terminals** | lowercase / quoted | Actual words that appear in sentences | `'buy'`, `'the'`, `'order'` |
| **Non-terminals** | UPPERCASE | Abstract categories that expand into other symbols | `S`, `VP`, `NP`, `V`, `N` |
| **Production rules** | `→` or `->` | Rules that define how non-terminals expand | `VP -> V NP` |
| **Start symbol** | `S` | The top-level symbol that represents a complete sentence | `S -> VP` |

## How to Read a Grammar Rule

```
VP -> V NP
```

This reads: *"A Verb Phrase (VP) consists of a Verb (V) followed by a Noun Phrase (NP)."*

```
NP -> Det N | N
```

The `|` means "or": *"A Noun Phrase is either a Determiner followed by a Noun, or just a Noun."*

## Standard Non-Terminal Abbreviations

| Symbol | Stands For | Example Words |
|--------|-----------|---------------|
| `S` | Sentence | (complete utterance) |
| `VP` | Verb Phrase | "buy the order", "cancel" |
| `NP` | Noun Phrase | "the order", "a subscription" |
| `PP` | Prepositional Phrase | "in the cart", "by Friday" |
| `V` | Verb | "buy", "cancel", "update" |
| `N` | Noun | "order", "subscription" |
| `Det` | Determiner | "the", "a", "an" |
| `Adj` | Adjective | "new", "active", "large" |
| `P` | Preposition | "in", "on", "by" |

---

# Part 3: Building and Using CFGs in NLTK

## Example 1: A Simple Command Grammar

Let's start with the example from the lecture — a grammar for parsing product action commands.

In [ ]:
import nltk

grammar = nltk.CFG.fromstring("""
  S -> VP
  VP -> V NP
  NP -> Det N | N
  Det -> 'the' | 'a'
  V -> 'buy' | 'cancel'
  N -> 'order' | 'subscription'
""")

parser = nltk.ChartParser(grammar)

sentence = "cancel subscription".split()
print(f"Parsing: {sentence}\n")

for tree in parser.parse(sentence):
    print(tree)

### Reading the Parse Tree

The output is a nested structure:

```
(S
  (VP
    (V cancel)
    (NP
      (N subscription))))
```

This tells us:
- The **sentence** (`S`) is a **verb phrase** (`VP`)
- The verb phrase consists of a **verb** (`V` = "cancel") and a **noun phrase** (`NP`)
- The noun phrase is a single **noun** (`N` = "subscription")

From this parse tree, an agent can deterministically extract: **action = cancel**, **object = subscription**.

In [ ]:
# Let's try several valid sentences
test_sentences = [
    "cancel subscription",
    "buy order",
    "cancel the order",
    "buy a subscription",
]

for sent_str in test_sentences:
    tokens = sent_str.split()
    parses = list(parser.parse(tokens))
    if parses:
        print(f"'{sent_str}' -> VALID")
        print(f"  {parses[0]}\n")
    else:
        print(f"'{sent_str}' -> REJECTED (no valid parse)\n")

In [ ]:
# Now try sentences that should be REJECTED by the grammar
invalid_sentences = [
    "destroy subscription",   # "destroy" is not in our vocabulary
    "cancel",                 # missing the noun phrase
    "the order cancel",       # wrong word order
    "buy the big order",      # "big" is not in our grammar
]

for sent_str in invalid_sentences:
    tokens = sent_str.split()
    parses = list(parser.parse(tokens))
    if parses:
        print(f"'{sent_str}' -> VALID (unexpected!)")
    else:
        print(f"'{sent_str}' -> REJECTED")

This is exactly the **hard constraint** property: the grammar only accepts commands that match its structure. "destroy subscription" is rejected because "destroy" is not a recognized verb, and "buy the big order" is rejected because there is no rule for adjectives.

## Example 2: Extracting Actions from Parse Trees

Parse trees are not just for display — they are **data structures** you can traverse programmatically. Let's extract the verb and object from a parsed command.

In [ ]:
def extract_command(tree):
    """Extract verb and object noun from a parsed command tree."""
    verb = None
    noun = None
    
    # Walk the tree to find V and N nodes
    for subtree in tree.subtrees():
        if subtree.label() == 'V':
            verb = subtree.leaves()[0]
        elif subtree.label() == 'N':
            noun = subtree.leaves()[0]
    
    return {'action': verb, 'object': noun}


# Parse and extract commands
commands = ["cancel the subscription", "buy a order", "cancel order"]

for cmd_str in commands:
    tokens = cmd_str.split()
    for tree in parser.parse(tokens):
        result = extract_command(tree)
        print(f"Input:  '{cmd_str}'")
        print(f"Parsed: action={result['action']}, object={result['object']}")
        print(f"  → Would call: {result['action']}_{result['object']}()\n")

This is the core of a **grammar-based command interpreter**: parse the input, extract structured fields, and map them to function calls. No LLM needed for this deterministic mapping.

## Example 3: A Richer Grammar with Adjectives and Prepositional Phrases

Real commands are more complex. Let's extend the grammar to handle adjectives and prepositional phrases like "add the new item to the cart".

In [ ]:
rich_grammar = nltk.CFG.fromstring("""
  S -> VP
  VP -> V NP | V NP PP
  NP -> Det N | Det Adj N | N | Adj N
  PP -> P NP
  Det -> 'the' | 'a' | 'my'
  Adj -> 'new' | 'active' | 'pending' | 'old'
  V -> 'add' | 'remove' | 'update' | 'cancel' | 'move'
  N -> 'item' | 'order' | 'cart' | 'subscription' | 'list' | 'account'
  P -> 'to' | 'from' | 'in'
""")

rich_parser = nltk.ChartParser(rich_grammar)

test_commands = [
    "add the new item to the cart",
    "remove old order from my list",
    "cancel active subscription",
    "update the pending order",
    "move item to my cart",
]

for cmd_str in test_commands:
    tokens = cmd_str.split()
    parses = list(rich_parser.parse(tokens))
    if parses:
        print(f"'{cmd_str}'")
        print(f"  {parses[0]}\n")
    else:
        print(f"'{cmd_str}' -> NO VALID PARSE\n")

### Visualizing Parse Trees

NLTK can draw graphical parse trees. This is very helpful for understanding complex structures.

In [ ]:
# Pretty-print a parse tree with indentation
# (NLTK's .draw() method opens a GUI window; .pretty_print() works in notebooks)

sentence = "add the new item to the cart".split()
for tree in rich_parser.parse(sentence):
    tree.pretty_print()

The tree shows the hierarchical structure clearly:
- The **verb phrase** (`VP`) has a verb, a noun phrase, and a prepositional phrase
- The **prepositional phrase** (`PP`) contains a preposition and another noun phrase
- Each **noun phrase** can include an optional determiner and adjective

## Example 4: Navigating Parse Trees Programmatically

NLTK `Tree` objects support several methods for navigation and inspection.

In [ ]:
sentence = "add the new item to the cart".split()
tree = list(rich_parser.parse(sentence))[0]

# Basic tree properties
print(f"Tree label (root): {tree.label()}")
print(f"Leaves (words):    {tree.leaves()}")
print(f"Height:            {tree.height()}")
print(f"Number of subtrees: {len(list(tree.subtrees()))}\n")

# List all subtrees and their labels
print("All subtrees:")
for subtree in tree.subtrees():
    print(f"  {subtree.label():<5} -> {subtree.leaves()}")

In [ ]:
def extract_rich_command(tree):
    """Extract structured command from a parse tree with PP support."""
    result = {'action': None, 'object': None, 'modifiers': [], 'destination': None}
    
    # Find the verb
    for subtree in tree.subtrees():
        if subtree.label() == 'V':
            result['action'] = subtree.leaves()[0]
            break
    
    # Find the VP to get its direct children
    for subtree in tree.subtrees():
        if subtree.label() == 'VP':
            children = list(subtree)
            for child in children:
                if hasattr(child, 'label'):
                    if child.label() == 'NP':
                        # Direct object NP
                        for np_child in child.subtrees():
                            if np_child.label() == 'N':
                                result['object'] = np_child.leaves()[0]
                            elif np_child.label() == 'Adj':
                                result['modifiers'].append(np_child.leaves()[0])
                    elif child.label() == 'PP':
                        # Prepositional phrase — extract preposition and destination
                        prep = None
                        dest = None
                        for pp_child in child.subtrees():
                            if pp_child.label() == 'P':
                                prep = pp_child.leaves()[0]
                            elif pp_child.label() == 'N':
                                dest = pp_child.leaves()[0]
                        result['destination'] = f"{prep} {dest}" if prep and dest else None
            break
    
    return result


# Test with several commands
commands = [
    "add the new item to the cart",
    "remove old order from my list",
    "cancel active subscription",
]

for cmd_str in commands:
    tokens = cmd_str.split()
    for tree in rich_parser.parse(tokens):
        cmd = extract_rich_command(tree)
        print(f"Input:    '{cmd_str}'")
        print(f"  action:      {cmd['action']}")
        print(f"  object:      {cmd['object']}")
        print(f"  modifiers:   {cmd['modifiers']}")
        print(f"  destination: {cmd['destination']}")
        print()

## Example 5: Ambiguous Grammars and Multiple Parses

An important concept in parsing: some sentences are **ambiguous** — they have more than one valid parse tree. This is not a bug; it reflects genuine ambiguity in language.

In [ ]:
# A grammar that allows PP attachment ambiguity
ambiguous_grammar = nltk.CFG.fromstring("""
  S -> NP VP
  VP -> V NP | V NP PP
  NP -> Det N | Det N PP
  PP -> P NP
  Det -> 'the' | 'a'
  V -> 'saw'
  N -> 'man' | 'dog' | 'telescope'
  P -> 'with'
""")

amb_parser = nltk.ChartParser(ambiguous_grammar)

# Classic ambiguous sentence: "the man saw the dog with the telescope"
# Does "with the telescope" modify how the man saw (VP attachment)
# or describe which dog (NP attachment)?
sentence = "the man saw the dog with the telescope".split()

parses = list(amb_parser.parse(sentence))
print(f"Found {len(parses)} valid parse(s) for: '{' '.join(sentence)}'\n")

for i, tree in enumerate(parses, 1):
    print(f"Parse {i}:")
    tree.pretty_print()
    print()

### Understanding the Ambiguity

**"The man saw the dog with the telescope"** has two readings:

1. **VP attachment**: The man used the telescope to see the dog
   - `VP -> V NP PP` ("with the telescope" modifies the seeing)

2. **NP attachment**: The man saw a dog that had a telescope
   - `NP -> Det N PP` ("with the telescope" modifies the dog)

Both parses are grammatically valid. In a real system, **disambiguation** would require context, world knowledge, or statistical preference — the kinds of things LLMs excel at. This is a great example of grammar + LLM complementarity.

## Example 6: Recursive Grammars

CFGs can handle **recursion** — rules that refer to themselves. This allows infinite nesting, like "the order in the cart on the table in the room".

In [ ]:
recursive_grammar = nltk.CFG.fromstring("""
  S -> NP VP
  VP -> V NP
  NP -> Det N | Det N PP
  PP -> P NP
  Det -> 'the'
  V -> 'found'
  N -> 'cat' | 'box' | 'table' | 'room'
  P -> 'in' | 'on' | 'under'
""")

rec_parser = nltk.ChartParser(recursive_grammar)

# Notice how NP -> Det N PP -> P NP creates recursion
# Each NP can contain a PP, which contains another NP, which can contain another PP...
sentences = [
    "the cat found the box",
    "the cat found the box on the table",
    "the cat found the box on the table in the room",
]

for sent_str in sentences:
    tokens = sent_str.split()
    parses = list(rec_parser.parse(tokens))
    if parses:
        print(f"'{sent_str}'")
        parses[0].pretty_print()
        print()

The recursion happens through the rule chain: `NP -> Det N PP` → `PP -> P NP` → `NP -> Det N PP` → ... This allows arbitrarily deep nesting, which is essential for modeling real language.

---

# Part 4: Grammar-Based Command Interpreter

Let's put everything together into a practical **command interpreter** — the kind of component an agentic system might use to parse user instructions.

In [ ]:
# Define a grammar for an e-commerce agent's command language
ecommerce_grammar = nltk.CFG.fromstring("""
  S -> VP
  VP -> V NP | V NP PP
  NP -> Det N | Det Adj N | N | Adj N | Num N | Num Adj N
  PP -> P NP
  Det -> 'the' | 'a' | 'my' | 'all'
  Adj -> 'new' | 'pending' | 'active' | 'expired' | 'recent'
  V -> 'cancel' | 'renew' | 'add' | 'remove' | 'list' | 'show' | 'update'
  N -> 'order' | 'orders' | 'subscription' | 'subscriptions' | 'cart' | 'item' | 'items' | 'account'
  P -> 'to' | 'from' | 'in'
  Num -> 'two' | 'three' | 'five'
""")

ecom_parser = nltk.ChartParser(ecommerce_grammar)


def interpret_command(sentence_str):
    """Parse a command and return a structured interpretation."""
    tokens = sentence_str.lower().split()
    parses = list(ecom_parser.parse(tokens))
    
    if not parses:
        return {'status': 'unrecognized', 'input': sentence_str}
    
    tree = parses[0]
    result = {
        'status': 'parsed',
        'input': sentence_str,
        'action': None,
        'target': None,
        'qualifier': None,
        'destination': None,
    }
    
    for subtree in tree.subtrees():
        if subtree.label() == 'V':
            result['action'] = subtree.leaves()[0]
    
    # Get the direct NP under VP
    for subtree in tree.subtrees():
        if subtree.label() == 'VP':
            for child in subtree:
                if hasattr(child, 'label') and child.label() == 'NP':
                    for np_child in child.subtrees():
                        if np_child.label() == 'N':
                            result['target'] = np_child.leaves()[0]
                        elif np_child.label() == 'Adj':
                            result['qualifier'] = np_child.leaves()[0]
                    break
                elif hasattr(child, 'label') and child.label() == 'PP':
                    for pp_child in child.subtrees():
                        if pp_child.label() == 'N':
                            result['destination'] = pp_child.leaves()[0]
            break
    
    return result


# Test the interpreter
commands = [
    "cancel my subscription",
    "show pending orders",
    "add new item to the cart",
    "renew active subscription",
    "remove expired items from my cart",
    "list all orders",
    "please do something weird",    # should fail
]

print(f"{'Command':<40} {'Status':<14} {'Action':<10} {'Target':<15} {'Qualifier':<12} {'Dest'}")
print("-" * 100)
for cmd in commands:
    r = interpret_command(cmd)
    if r['status'] == 'parsed':
        print(f"{cmd:<40} {r['status']:<14} {str(r['action']):<10} {str(r['target']):<15} {str(r['qualifier']):<12} {str(r['destination'])}")
    else:
        print(f"{cmd:<40} {r['status']:<14} → Would escalate to LLM")

The grammar-parsed commands go directly to tool calls. The unrecognized command ("please do something weird") would be **escalated to an LLM** for interpretation. This two-tier architecture — grammar for structured commands, LLM for everything else — is efficient and reliable.

---

# Part 5: Logic Programming and Prolog (DCGs)

## What Is Prolog?

**Prolog** is a logic programming language designed for symbolic reasoning. Unlike Python (which is imperative — you tell it *how* to compute), Prolog is declarative — you tell it *what* is true, and it figures out the rest.

Prolog is relevant to NLP because it supports **Definite Clause Grammars (DCGs)** — a notation for writing grammars that is built directly into the language.

## Definite Clause Grammars (DCGs)

A DCG is a way to write context-free grammars in Prolog syntax. Compare the same grammar in NLTK vs. Prolog:

**NLTK CFG:**
```python
S -> VP
VP -> V NP
NP -> N
V -> 'buy' | 'cancel'
N -> 'order' | 'subscription'
```

**Prolog DCG:**
```prolog
sentence --> verb_phrase.
verb_phrase --> verb, noun_phrase.
noun_phrase --> noun.
verb --> [buy] ; [cancel].
noun --> [order] ; [subscription].
```

Key syntactic differences:

| Feature | NLTK CFG | Prolog DCG |
|---------|----------|------------|
| Rule arrow | `->` | `-->` |
| Sequence | space-separated | comma-separated |
| Alternatives | `\|` | `;` |
| Terminals | quoted strings `'buy'` | bracketed atoms `[buy]` |
| Rule end | (none) | `.` (period) |

## What Makes Prolog Powerful for Agents

Beyond just parsing, Prolog can **enforce business rules** over parsed structures. Here's a conceptual example:

```prolog
% Grammar rules
sentence --> verb_phrase.
verb_phrase --> verb, noun_phrase.
verb --> [buy] ; [cancel].
noun_phrase --> [order] ; [subscription].

% Business constraint rules
allowed(cancel, subscription) :- subscription_status(active).
allowed(cancel, order) :- order_status(pending).
allowed(buy, subscription) :- not(subscription_status(active)).
```

This combines **parsing** (is the command grammatically valid?) with **reasoning** (is the action *allowed* given the current state?). In an agentic system:

1. The **grammar** parses the user's command into a structured form
2. The **constraint rules** check whether the action is allowed
3. If allowed → execute the action
4. If not → explain why ("You can only cancel active subscriptions")
5. The **LLM** sits on top for natural language understanding and response generation

## Simulating DCG-Style Parsing in Python

While we don't run Prolog here, we can simulate the same constraint-enforcement pattern in Python to illustrate the concept.

In [ ]:
# Simulated system state (like Prolog facts)
system_state = {
    'subscription_status': 'active',
    'order_1234_status': 'shipped',
    'order_5678_status': 'pending',
}

# Business constraint rules (like Prolog rules)
def is_allowed(action, target, state):
    """Check if a parsed command is allowed given the current state."""
    if action == 'cancel' and target == 'subscription':
        if state.get('subscription_status') == 'active':
            return True, "Subscription is active — cancellation allowed."
        else:
            return False, "Cannot cancel: subscription is not active."
    
    elif action == 'cancel' and target == 'order':
        # Check if any order is in a cancellable state
        pending = [k for k, v in state.items() if k.endswith('_status') and v == 'pending' and 'order' in k]
        if pending:
            return True, f"Found pending order(s) — cancellation allowed."
        else:
            return False, "Cannot cancel: no orders are in pending status."
    
    elif action == 'renew' and target == 'subscription':
        if state.get('subscription_status') in ('expired', 'cancelled'):
            return True, "Subscription can be renewed."
        else:
            return False, f"Cannot renew: subscription is {state.get('subscription_status')}."
    
    return True, "Action allowed (no specific constraints)."


# Parse with grammar, then check constraints
test_commands = [
    "cancel my subscription",
    "renew active subscription",
    "cancel pending orders",
]

print("Grammar Parse + Constraint Check:\n")
for cmd in test_commands:
    parsed = interpret_command(cmd)
    if parsed['status'] == 'parsed':
        allowed, reason = is_allowed(parsed['action'], parsed['target'], system_state)
        status = "ALLOWED" if allowed else "BLOCKED"
        print(f"  Command: '{cmd}'")
        print(f"  Parsed:  action={parsed['action']}, target={parsed['target']}")
        print(f"  Result:  {status} — {reason}\n")
    else:
        print(f"  Command: '{cmd}' → Could not parse\n")

This is the same pattern that Prolog handles natively: **grammar rules** for parsing + **logical rules** for constraint enforcement. Prolog's advantage is that both are expressed in the same declarative language, making the system easier to maintain and reason about.

---

# Part 6: The Big Picture — Grammar + LLM Architecture

```
┌───────────────────────────────────────────────────────────────────┐
│                   USER INPUT                                      │
│        "I want to cancel my subscription"                         │
└───────────────────────┬───────────────────────────────────────────┘
                        ↓
┌───────────────────────────────────────────────────────────────────┐
│              LLM NORMALIZATION LAYER (optional)                    │
│  Paraphrases → canonical form: "cancel my subscription"           │
└───────────────────────┬───────────────────────────────────────────┘
                        ↓
┌───────────────────────────────────────────────────────────────────┐
│              CFG PARSER (deterministic)                            │
│  Input:  ["cancel", "my", "subscription"]                         │
│  Output: action=cancel, target=subscription                       │
│                                                                   │
│  ✓ Parsed → proceed to constraint check                          │
│  ✗ No parse → escalate to LLM for interpretation                 │
└───────────────────────┬───────────────────────────────────────────┘
                        ↓
┌───────────────────────────────────────────────────────────────────┐
│              CONSTRAINT ENGINE (deterministic)                     │
│  Check: is cancel(subscription) allowed?                          │
│  State: subscription_status = active → YES                        │
└───────────────────────┬───────────────────────────────────────────┘
                        ↓
┌───────────────────────────────────────────────────────────────────┐
│              EXECUTE ACTION                                       │
│  cancel_subscription()                                            │
└───────────────────────────────────────────────────────────────────┘
```

The key insight: **deterministic parsing guarantees correctness** for the structured command subset, while **LLMs provide flexibility** for everything else.

---

# Practice Exercises

Try these exercises to reinforce what you've learned.

## Exercise 1: Extend the Grammar

Add support for the following commands to the `ecommerce_grammar`:
- "ship my order" / "ship the pending order"
- "apply a discount to the order"

You'll need to add new verbs and nouns.

In [ ]:
# TODO: Create an extended grammar that handles the new commands
# Then parse these test sentences:

test_sentences = [
    "ship my order",
    "ship the pending order",
    "apply a discount to the order",
]

# Your code here

## Exercise 2: Count the Parses

Using the ambiguous grammar from Example 5, determine how many parses the following sentence has. Then explain what each parse means in plain English.

Sentence: `"the man saw the dog with the telescope"`

In [ ]:
# TODO: Parse the sentence and pretty-print each parse tree
# For each parse, write a comment explaining the interpretation

# Your code here

## Exercise 3: Build a Query Grammar

Design a CFG for a simple data query language that accepts commands like:
- "show all customers"
- "find recent orders"
- "count active subscriptions"
- "show orders from the database"

Then write a function that extracts the query operation, target, and any filters.

In [ ]:
# TODO: 
# 1. Define a CFG for the query language
# 2. Parse the test sentences
# 3. Extract structured query parameters (operation, target, filter)

# Your code here

---

## Summary

| Concept | What It Does | Key Takeaway |
|---------|-------------|-------------|
| **Context-Free Grammar (CFG)** | Defines valid sentence structures via production rules | Provides hard constraints — only valid structures are accepted |
| **NLTK ChartParser** | Parses sentences against a CFG and produces parse trees | Deterministic, fast, explainable parsing |
| **Parse trees** | Hierarchical representation of sentence structure | Can be traversed to extract actions, objects, modifiers |
| **Ambiguity** | One sentence, multiple valid parse trees | Reflects real language; disambiguation needs context or statistics |
| **Recursion** | Grammar rules that refer to themselves | Enables unbounded nesting (PP within NP within PP...) |
| **Prolog DCGs** | Grammars written in logic programming notation | Combine parsing with constraint reasoning in one formalism |

### Key Takeaways

1. **CFGs provide hard constraints**: only inputs matching the grammar are accepted — no hallucination, no approximation
2. **Parse trees are data structures**: traverse them programmatically to extract structured information
3. **Ambiguity is inherent** in natural language — grammars expose it explicitly, which is valuable for debugging
4. **Prolog DCGs** unify grammar parsing with logical constraint enforcement — a powerful pattern for agent systems
5. **Grammar + LLM** is a powerful architecture: use grammars for structured commands, LLMs for flexibility and fallback